![title](asci_art.png)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import binned_statistic
from scipy.special import i0e, k0e, i1e, k1e
from matplotlib.patches import Rectangle

# --- DATA PREPARATION ---
# (Re-using the data arrays from previous steps. Ensure they are loaded.)
# If you need to reload, please re-run the "Data Loading Cell" from the history.
# Assuming 'r_dat', 'v_dat', 'e_dat' exist. 
# For safety in this standalone block, I will check if they exist, if not
# you must run the data loading cell first. 
if 'r_dat' not in locals():
    print("WARNING: r_dat not found. Please run the Data Loading Cell first.")
    # Placeholder for syntax checking if run without data
    r_dat = np.logspace(-1, 2.5, 100)
    v_dat = np.zeros_like(r_dat)
    e_dat = np.ones_like(r_dat)

# Binning Function
def bin_data(r, v, e, bins=40):
    bin_edges = np.logspace(np.log10(min(r)), np.log10(max(r)), bins + 1)
    r_bin, _, _ = binned_statistic(r, r, statistic='mean', bins=bin_edges)
    v_bin, _, _ = binned_statistic(r, v, statistic='mean', bins=bin_edges)
    def error_stat(x): return np.sqrt(np.sum(x**2)) / len(x)
    e_bin, _, _ = binned_statistic(r, e, statistic=error_stat, bins=bin_edges)
    mask = ~np.isnan(r_bin)
    return r_bin[mask], v_bin[mask], e_bin[mask]

r_binned, v_binned, e_binned = bin_data(r_dat, v_dat, e_dat, bins=45)

# Weighting Strategy (Inflate outer errors to avoid overfitting scatter)
weights_fit = e_binned.copy()
mask_outer = r_binned > 25.0
weights_fit[mask_outer] *= 2.0 

# --- PHYSICS MODELS ---

def safe_exp(x):
    return np.exp(np.clip(x, -200, 200))

def newtonian_disk_velocity(r, M_disk=6.5e10, r_d=3.5):
    G = 4.300e-6 
    y = r / (2 * (r_d + 1e-6))
    y = np.maximum(y, 1e-9)
    bessel = i0e(y) * k0e(y) - i1e(y) * k1e(y)
    return np.sqrt((2 * G * M_disk / r_d) * (y**2) * np.maximum(bessel, 0))

def newtonian_bulge_velocity(r, M_bulge=0.9e10, r_b=0.5):
    G = 4.300e-6
    return np.sqrt(G * M_bulge * r / ((r + r_b)**2 + 1e-6))

def gm_velocity_component(r, B_BH, r_bh, B_Disk, r_c, k_cluster, r_cut):
    # 1. SMBH Vortex
    term1 = B_BH / (r**2 + r_bh**2)**1.5
    # 2. Disk Vortex
    term2 = B_Disk * (1 - safe_exp(-r/(r_c+1e-6)))**2 / (r**2 + 0.1)
    # 3. Cluster Floor
    term3 = k_cluster * safe_exp(-r / (r_cut+1e-6))
    
    k_total = term1 + term2 + term3
    return r * k_total

def full_model(r, B_BH, r_bh, B_Disk, r_c, k_cluster, r_cut):
    v_bar_sq = newtonian_disk_velocity(r)**2 + newtonian_bulge_velocity(r)**2
    v_gm = gm_velocity_component(r, B_BH, r_bh, B_Disk, r_c, k_cluster, r_cut)
    return (v_gm + np.sqrt(v_gm**2 + 4 * v_bar_sq)) / 2

def no_cluster_model(r, B_BH, r_bh, B_Disk, r_c):
    # Forces k_cluster = 0
    return full_model(r, B_BH, r_bh, B_Disk, r_c, 0.0, 100.0)

def get_red_chisq(model_func, popt):
    v_m = model_func(r_binned, *popt)
    chisq = np.sum(((v_binned - v_m)/e_binned)**2)
    return chisq / (len(r_binned) - len(popt))

# --- FITTING ROUTINE ---

def run_comparison():
    # 1. Fit FULL Model
    # Params: B_BH, r_bh, B_Disk, r_c, k_cluster, r_cut
    p0_full = [1e4, 0.5, 2500.0, 5.0, 2.0, 120.0]
    bounds_full = ([0, 0.01, 0, 0.1, 0, 50.0], [np.inf, 5.0, np.inf, 15.0, 10.0, 500.0])
    
    popt_full, pcov_full = curve_fit(full_model, r_binned, v_binned, sigma=weights_fit, absolute_sigma=True, p0=p0_full, bounds=bounds_full, maxfev=50000)

    # 2. Fit NO CLUSTER Model
    # Params: B_BH, r_bh, B_Disk, r_c
    p0_nc = [1e4, 0.5, 2500.0, 5.0]
    bounds_nc = ([0, 0.01, 0, 0.1], [np.inf, 5.0, np.inf, 50.0]) 
    
    popt_nc, pcov_nc = curve_fit(no_cluster_model, r_binned, v_binned, sigma=weights_fit, absolute_sigma=True, p0=p0_nc, bounds=bounds_nc, maxfev=50000)

    # --- PLOTTING ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 14), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    r_plot = np.logspace(np.log10(min(r_dat)), np.log10(max(r_dat)), 1000)

    # --- BACKGROUND ZONES (Bulge, Disk, Halo) ---
    # Define regions based on typical scales (approximate)
    ax1.axvspan(0.01, 3.0, color='gray', alpha=0.1, label='Bulge Region')
    ax1.axvspan(3.0, 15.0, color='lightblue', alpha=0.1, label='Disk Region')
    ax1.axvspan(15.0, 200.0, color='lavender', alpha=0.1, label='Halo Region')

    # A. Plot Data
    # Full data with low alpha
    ax1.errorbar(r_dat, v_dat, yerr=e_dat, fmt='.', color='gray', alpha=0.25, zorder=0, label='Full Data (Raw)')
    # Binned data
    ax1.errorbar(r_binned, v_binned, yerr=e_binned, fmt='o', color='black', alpha=0.8, label='Binned Data', zorder=5)

    # B. Plot Full Model (with Confidence)
    v_full = full_model(r_plot, *popt_full)
    
    # MC Error Prop for Full Model
    n_samples = 200
    v_samples = np.zeros((n_samples, len(r_plot)))
    params_mc = np.random.multivariate_normal(popt_full, pcov_full, n_samples)
    for i in range(n_samples):
        v_samples[i,:] = full_model(r_plot, *params_mc[i])
    
    v_1sig = np.percentile(v_samples, [16, 84], axis=0)
    v_2sig = np.percentile(v_samples, [2.5, 97.5], axis=0)
    
    ax1.fill_between(r_plot, v_2sig[0], v_2sig[1], color='red', alpha=0.1, label='Full Model 2$\sigma$')
    ax1.fill_between(r_plot, v_1sig[0], v_1sig[1], color='red', alpha=0.2, label='Full Model 1$\sigma$')
    ax1.plot(r_plot, v_full, 'r-', lw=3, zorder=10, label=f'Full Model (Red. $\chi^2$={get_red_chisq(full_model, popt_full):.2f})')

    # C. Plot No Cluster Model
    v_nc = no_cluster_model(r_plot, *popt_nc)
    ax1.plot(r_plot, v_nc, 'b--', lw=2, label=f'No Intergalactic Term (Red. $\chi^2$={get_red_chisq(no_cluster_model, popt_nc):.2f})')

    # D. Plot Components (For Full Model)
    # Baryons
    v_bar_disk = newtonian_disk_velocity(r_plot)
    v_bar_bulge = newtonian_bulge_velocity(r_plot)
    v_bar_total = np.sqrt(v_bar_disk**2 + v_bar_bulge**2)
    
    ax1.plot(r_plot, v_bar_disk, color='blue', linestyle=':', alpha=0.6, label='Baryon Disk')
    ax1.plot(r_plot, v_bar_bulge, color='green', linestyle=':', alpha=0.6, label='Baryon Bulge')
    ax1.plot(r_plot, v_bar_total, color='navy', linestyle='--', lw=1.5, label='Total Baryonic')
    
    # GM Components
    # Recalculate individual GM components using fitted parameters
    # popt_full = [B_BH, r_bh, B_Disk, r_c, k_cluster, r_cut]
    B_BH_fit, r_bh_fit, B_Disk_fit, r_c_fit, k_cl_fit, r_cut_fit = popt_full
    
    # GM Core (SMBH)
    v_gm_core = r_plot * (B_BH_fit / (r_plot**2 + r_bh_fit**2)**1.5)
    # GM Disk Vortex
    v_gm_disk = r_plot * (B_Disk_fit * (1 - safe_exp(-r_plot/(r_c_fit+1e-6)))**2 / (r_plot**2 + 0.1))
    # GM Cluster
    v_gm_clus = r_plot * (k_cl_fit * safe_exp(-r_plot / (r_cut_fit+1e-6)))
    # Total GM
    v_gm_tot = gm_velocity_component(r_plot, *popt_full)
    
    ax1.plot(r_plot, v_gm_core, color='orange', linestyle='-.', alpha=0.7, label='GM Core (SMBH + Bulge)')
    ax1.plot(r_plot, v_gm_disk, color='cyan', linestyle='-.', alpha=0.7, label='GM Disk')
    ax1.plot(r_plot, v_gm_clus, color='purple', linestyle='-.', alpha=0.7, label='GM Intergalactic')
    ax1.plot(r_plot, v_gm_tot, color='magenta', linestyle='-', lw=2, alpha=0.8, label='Total Gravitomagnetic Field')

    # E. Plot Characteristic Radii
    # r_bh, r_c, r_cut
    # We plot vertical lines or markers
    # --- ADD CENTRAL REGION MARKERS ---
    
    # 1. SMBH Sphere of Influence (r_SOI) ~ 2 pc
    # Already calculated, but defining explicitly for clarity
    # r_soi = 0.002  # kpc (2 pc)
    # ax1.axvline(r_soi, color='black', linestyle='--', linewidth=1.5, alpha=0.9)
    # ax1.text(r_soi * 1.2, 50, r'$r_{\text{SOI}}$ (SMBH)', color='black', 
    #          rotation=90, verticalalignment='bottom', fontsize=10, fontweight='bold')

    # 2. Circumnuclear Disk (CND) ~ 5 pc
    # The torus of neutral gas/dust immediately surrounding the nucleus
    # r_cnd = 0.005  # kpc (5 pc)
    # ax1.axvline(r_cnd, color='darkred', linestyle='-.', linewidth=1.2, alpha=0.7)
    # ax1.text(r_cnd * 1.2, 280, r'CND (5 pc)', color='darkred', 
    #          rotation=90, verticalalignment='top', fontsize=9)

    # 3. Central Molecular Zone (CMZ) ~ 150 pc
    # The "Molecular Region" with radio arcs and massive clouds
    r_cmz = 0.150  # kpc (150 pc)
    ax1.axvline(r_cmz, color='k', linestyle='-.', linewidth=1.2, alpha=0.7, label='Central Molecular Zone (CMZ)')
    ax1.text(r_cmz * 1.1, 70, r'CMZ (150 pc)', color='k', 
             rotation=90, verticalalignment='top', fontsize=11)

    
    ylim_ax1 = ax1.get_ylim()
    # r_bh
    ax1.axvline(r_bh_fit, color='k', linestyle=':', alpha=0.5)
    ax1.text(r_bh_fit, 20, r'$r_{Core}$', color='k', rotation=90, verticalalignment='bottom', fontsize=18)
    # r_c
    ax1.axvline(r_c_fit, color='k', linestyle=':', alpha=0.5)
    ax1.text(r_c_fit, 20, r'$r_{Disk}$', color='k', rotation=90, verticalalignment='bottom', fontsize=18)
    # r_cut
    ax1.axvline(r_cut_fit, color='k', linestyle=':', alpha=0.5)
    ax1.text(r_cut_fit, 20, r'$r_{Hill}$', color='k', rotation=90, verticalalignment='bottom', fontsize=18)



    # --- HIGHLIGHT BOX FOR OU ET AL. DATA ---
    # Calculate bounds dynamically from the data array
    r_ou_min, r_ou_max = np.min(ou_2024[:, 0]), np.max(ou_2024[:, 0])
    v_ou_min, v_ou_max = np.min(ou_2024[:, 1]), np.max(ou_2024[:, 1])
    
    # Add padding (10% on radius, 20 km/s on velocity)
    pad_r = (r_ou_max - r_ou_min) * 0.1
    pad_v = 20.0 
    
    # Create the rectangle anchor (bottom-left x, y) and dimensions
    anchor = (r_ou_min - pad_r, v_ou_min - pad_v)
    width = (r_ou_max - r_ou_min) + 2 * pad_r
    height = (v_ou_max - v_ou_min) + 2 * pad_v
    
    # Create and add the patch
    # zorder=0 puts it behind the data points but in front of the grid
    ou_box = Rectangle(anchor, width, height, 
                       linewidth=2, edgecolor='green', facecolor='none', 
                       linestyle='-', zorder=5, label='High-Precision Region (Gaia/APOGEE)', alpha=.55)
    
    ax1.add_patch(ou_box)
    
    # Add a text annotation for clarity (Optional)
    ax1.text(anchor[0], anchor[1] + height + 5, "Ou et al. (2024)", 
             color='green', fontsize=10, fontweight='bold')

    
    ax1.set_ylabel('Velocity (km/s)', fontsize=14)
    ax1.legend(loc='upper right', ncol=3, fontsize=12.5, framealpha=0.9)
    ax1.set_ylim(0, 350)
    ax1.set_xlim(0.02, 200)
    ax1.grid(True, alpha=0.3)

    # --- Residuals with Shading ---
    res_full = v_binned - full_model(r_binned, *popt_full)
    res_nc = v_binned - no_cluster_model(r_binned, *popt_nc)
    
    # Calculate model uncertainty for residuals
    # We propagate the uncertainty of the model to the residual plot (centered at 0)
    v_samples_binned = np.zeros((n_samples, len(r_binned)))
    for i in range(n_samples):
        v_samples_binned[i,:] = full_model(r_binned, *params_mc[i])
    
    res_std = np.std(v_samples_binned, axis=0)
    
    # Plot 1 and 2 sigma bands for residuals
    # Interpolate for smooth plotting
    res_std_plot = np.interp(r_plot, r_binned, res_std)
    
    ax2.fill_between(r_plot, -2*res_std_plot, 2*res_std_plot, color='red', alpha=0.1, label='2$\sigma$ Model Error')
    ax2.fill_between(r_plot, -1*res_std_plot, 1*res_std_plot, color='red', alpha=0.2, label='1$\sigma$ Model Error')

    ax2.errorbar(r_binned, res_full, yerr=e_binned, fmt='o', color='red', alpha=0.8, label='Full Model Residuals')
    ax2.plot(r_binned, res_nc, 'bx', alpha=0.6, label='No Intergalactic Residuals')
    ax2.axhline(0, color='black', ls='--')
    
    ax2.set_ylabel('Residuals (km/s)', fontsize=14)
    ax2.set_xlabel('Radius (kpc)', fontsize=14)
    ax2.set_xscale('log')
    ax2.legend(fontsize=12.5)
    ax2.grid(True, linestyle='dotted', alpha=0.5)

    plt.tight_layout()
    fig.savefig('galaxy_curve_fit_detailed.jpeg', dpi=150)
    plt.show()
    
    print("Compare Parameters:")
    print(f"Full Model r_c: {popt_full[3]:.2f} kpc")
    print(f"No Cluster r_c: {popt_nc[3]:.2f} kpc")
    print(f"Fit Parameters (Full): {popt_full}")

run_comparison()

In [2]:
..  .                                                                                                                                                                                                                                                                                                                                                                   .                        .            ..
.. ....                                                                                                                                                                                                                                                                                                                     .                                                                    ..     ..   ...
.. .@-.. ....  .   .                                                                            .   .    .  .         .        .                                            .                                                 ..     .                                 .       .  .                                         .                                                                .   ...  ..@:    ..
 .%@@@@@@@@@#@@@@@@@@@@#@@@@@@@@@@#@@@@@@@@@@@@@@@@@@@@@*@@@@@@@@@@@@@@@@@@@@@*@@@@*@@@@@@@@@@@@@@@@@@@@@%@@@@@@@@@@*@@@@#@@@@@@@@@@@@@@@@@@@@@*@@@@@@@@@@%@@@@@@@@@@%@@@@@@@@@@#@@@@@@@@@@%@@@@@@@@@@*@@@@#@@@@@@@@@@@@@@@@#@@@@#@@@@@@@@@@@@@@@@#@@@@*@@@@@@@@@@@@@@@@%@@@@*@@@@@@@@@@@@@@@@@@@@@*@@@@@@@@@@*@@@@@@@@@@@@@@@@@@@@@*@@@@*@@@@@@@@@@#@@@@@@@@@@#@@@@@@@@@@#@@@@@@@@@@%@@@@@@@@@@#@@@@@@@@@@.....
   .*:                                                                                                                                                                                                                                                                                                                                                                                                  #.......
..                                                                                                                                                                                                                                                                                                                                                                                                         .....
 .  +:                                                                                                                                                                                                                                                                                                                                                                                                  +:.. ...
..  @=                                                           .                                                                                                 .                                                                                                                                      .                                                                 .                           @=  ....
.. .@=                                                                                                                                                                                                                                                                                                                                       .                                                          @=  . ..
..  @=                                                    .                                                                                                                                                                                                                                                                                                                                             @=   ...
.   @=                                                                                                                                                                                                                                                                .                                                                                                                                 @=   ...
.                                               .                                                                                                                                                                                                                                              .                                                                                                        .    ...
    @=                                                                                                   .                                                                                                                                                           .           .                                                                                                                      @=     .
    @=                                                                                                                                                                       .                                                                                                                             .                                                                                            @=.  ...
    @=               .                                                                                                                                                                                                                                     .     .                                                                                                                                      @=. ....
    @=                                                            .                                      ...........................          .......                     .......               .......               ............                .............            ................................                                                                                             @=.  ...
    @=                                                                                                   .........:..:...:::........   .      ....:::.                    .::....              .:::....               ....:::.....                 ............            ......::.::::::.::::......:::...                                                                                             @=     .
.   -.                                                                                                 ..                             =:...+.         .+.             .+.          +:       ::          .+        -.             .:              :              .+      =.                                  .=                                                                                          =. .   .
.   @=                                                                                                :@-                           ..@+  .@:         .@.             .@:          @+       ##          =@.       @+              @#            *@              -@      @:                                  =@                                                                                          @=    ..
    @=                                                                                               .@+                            ..@=  .@:         .@.             .@-          @*       #%          =@.       @+              .@+          =@.              -@     .@-                                  =@            .                                                   .                         @=   ...
    @=                                                                                               @@.                            ..@=  .@:         .@:             .@:          @+      .#%.         -@        @+.              .@-        :@.               -@.    .@-                                  =@                                                                                          @=    ..
    @=                                                                                              =@.        -################****..@+...@:         .@:.            .@:          @+      .#%          -@.   .   @+                -@       .@=                -@.   ..@-            .*##############*#*#+.=@                                                                                          @=    ..
    *:                                                                                           .                    .               *-  .@.         .@..             %.         .%:       ==          :#    ....*-                                            .#     .#.         .          .....       . :#                                                                                          +:   ...
    @=                                                                                    ..    @*          .-@.                          .@:     . ..:@:.            .@:     .....@*..     #%          -@..  .. .@+                 -@.     @-                 -@     .@:          @-                                                                                                                  @=   ...
    @=                                                                                          %*         .:@.                           .@.     .....@:             .@:    .    .@*.  .   #%          =@.       @+                  #%.   #%                  -@      @:          @-                                                                                                                  @=.  ...
    @=                 .                  .                                                    .%*         .@+.                           .@.     .   .@.             .@:         .@*..     *#          =@.       @=                  .@%..*@.                  -@      @:          @-                                .                                                                                 @=. ....
    @=                                                                                          @*        .@@                             .@:.        .@.             .@-         .@*      .##          =@.       @+.               . ..@=-@.                  .=@.     @:          @-                                                                  .                                               @=  ....
.   @=                                      .                                                   @*        .=                              .@:         .@: +++++++++++..@-          @*     ..#%.         =@.      .@+.                   .::.                    -@.     @:          @-.+++++**+++.                                                                      .                               @=.. ...
                                                                                                .                                          .           .               ..                  ...          ..       ...                                            ..      .           .                               .                                                                                   .. .  ..
    @=                                                                                          %+          @-                            .@:                                      @+      .#%.         =@.       @+          -@ .@*            =@. *%        . -@      @:                         @+                                                                                                   @=.   ..
.   @=                                                                                          %+         .@-                            .@:                .                     @+       *#          =@.       @=          -@. :@:          :@-  *%          -@      @:                         @+                                                                                                   @=.    .
.   @=                                                                                          %+         .@-                            .@.                                     .@*       *#          =@.       @=          -@...:@.        .@-. .*%          -@.     @:                        .@+                                                              .      .                             @=    ..
..  @=                                                                                         .%+         .@-                            .@.                                     .@#.      *#          =@.       @=          -@    =@.       @*    +%          -@.     @:                         @+       .                                                                                           @=    ..
.   %-                                                                                         .*=         .@:                            .@.             =+++*****++.            .@+.      =+.         -@.       #=..        :@                    -*          :@..    @:            .+*****++*+. @=                                                                                                   %-.   ..
.   +.                                                                                          --        .=                               %.         .@.             .%:          *-      .-=.    .    :*       .+-..        :+     .+.     =..    :=          .*      #.          %:                                            .                                                                     *:.  ...
.  .@=                                                       .    .                             %*. .      @@                             .@:         .@:             .@:          @+      .#%.         -@       .@+.         -@     .@#.   =@.    .*%          -@      @- .        @-    .                                                                                                             @=......
.   @=                                                         .                                %*         .@+                            .@.         .@:             .@:          @+      .#%          -@        @=          -@      .@-  .@.     .*%          -@      @-          @-                                                   .                   ..                                         @=. ....
.   @=                                                                                          %*.  .      :@.    .                      .@.         .@:             .@:          @+      .#%          -@.       @+          -@.      .@:.@:       *%.         -@      @:          @:                                                                                                                  @= .....
.   @=                                                                                          %+           :%#@@@@@@@@@@@@@@@@@@@@:     .@:         .@:             .@:         .@*      .#%          -@.       @+          -@.       -#%=        *%          -@      @:          @-:@@@@@@@@@@@@@@@@@@@@                                                                                             @=  ....
                                                                                                                                          ...                                 ..            ..          ...    .              ...                                 .                 ....                                                                                                                     ...
.  .@=                                                                                              =@          .                    .@=. .@.         .@:             .@:    . .  .@*       *#          =@.       @=          -@                    *%          -@..   .@:     .                            -@                                              .                                           @=    ..
    @=                                                                                              .@@                              .@=  .@.         .@:             .@:         .@*       *#..        =@.       @+.         -@                    *%          -@.     @:                                  =@                                                                                          @=    ..
    @=                                          .                                                     @=                              @=  .@:.        .@:             .@:.         @+  . .  **          -@        @=    .     -@                    *%.         -@. .   @-                                  -@                                                                                          @=  ....
   .@=                                                                                                :@:.                           .@+...@.         .@:             .@:          @*.     .##          -@........@=    ..    -@                    *%          -@      @:                                  -@                                             .                                            @=  ....
    @=                                 .                        .                                      :.-##########################..@=...@. :#####*..@:             .@: :#####* .@*.     .#%..######: -@.       @+..######. -@                    *%  +%%%%%= -@      @: +##############################+.=@                                                                                          @=   ...
    =.                                                                                                                                                                            .                                                        ..   .                                                  .                                  ...                                                               -. .....
....@=                                                                                                                                                    .                                                                                                                                                                                                                                             @= .....
 ...@=                          .......                   .....                                                                                                                    .   .                    ..     .    .                                                                                                 ....                                                           .....    . .   @=......
.. .@=                                              .. . .                                                                                                                 .. ..  ....                                                                                                                                     ........                                                      ...            @=......
.. .@=                                                  ..                                                                                                 .            ....        .                                                                                                                                                                                                     ..            @=. ....
    -.                                        .         .                                                                                      .                  .  .                                  ..                                                                                                                                                                                  .           +:......
.   @=                                                                                                                                                          ..       .                                                                                                                                                                               .                                              @=......
    @=                                            .                                                                                                                                             .                                                                                                                      .                                                                                @=......
.   @=             .                                                                                                                                        .                                                                                                                                                                                                                                           @=.  ...
  . @=                                                                                                                                                                                                                                                                                                                                                                                                  @=  ....
   .@=      ..                                                      ..                                                             .                       ......                   .... .. .                              ..                  . .... ..                   .                                                         ...  .                                                   .  . ..   @=    ..
    ..        ..                   .                                                                                                                                           .     ... .@@@+:.....                  .  .                                                                                                             .               .        ...                                     .   ....
    @=       ..                                                .                            .                          ...  .                            :%@@@#=:.   .    ..  .   .  ....@@%%@%#**%@@@@@@#=:.....   .                             .              . ... .   ..                                                   ....            .            ... ...                            .       @=......
....@=        ..  .                                     ..........    .                 .. .....                  .-@@%*-...             ...     .   .  .=@@@@%+*%@@@@@#=-:.::%@@%+=-:..#=.....  .-#@@@@%*+=+#@@@@@@%+-:..           . .........  .                ...........                .......                        .........              . .....  .  ..                          ..          @=......
....@=       . .                           ..       .. . .. ...   .*@@. ...     ..    . .. .  .                  ..@@@@-:=@@@%. .    ...*:=#:@@+        +@.......:#@@@@%-.-#@@@@@+:...#..@#@#:    ... =+ ....-*@@@@@#:..-+@@@@@@+:  ........ .    .... .   .    ... ..... . .   .    .  .  .    .....                             ..          .    ..    .                                              @=......
....@=                                                  .  .   .  :@.-%@@:                 .=..:@@:           ....@*+..:=#@@%-:-#@@@#:.-.        *--*@%#%-..     ..  -%.:-+@@@@@#=-=*@@@@@*: .:.-==+#@%+.....    ....:-=*%@@@@@%=-:-=+#@@@@@#-.                     .                .    .. .....            .                    .           .  ..         .                 .                        @=......
....*:                      ..                    . ..........   .+#@@@-:%@@+.. . .     .. --: .#%.+@@+.      ...@@-         -*@@@@*-+@@@@%=.       ....-@%*##+.    :@...........-*@@@@@%+::+@@@@@@%*-....:+****#+:....  =#. ....:=#@@@@@@%*=::+%@@@@@@%*=:.  .     .                    .......                                                ..        .                . ......                     *-......
.. .@-           .                      .*@=.:.      .          .-@#:..-@%:.#@@#:..       #.-.:=-:= .**.#@#:... +@...       #-..:::-%@@@#-..+@@@%+-......  ..   .@@@@+-:.....     .....=*.=%@@@@@#=:.-*@@@@@#==:.:...-*@@#-::.            ...-*@@@@@%*+-:..-#@@@@@@*=-:..        .:--:.....   .                                               ...                                                       @-......
....@-                      .             .#*@:%%.   . .     . .+@.@... :#.@@..@@@%.   ::....---..  :-+=..**:@@%@.*        @*..:+:...    .*@@@%:.::%@@@*.         ......:#@*:    ...  :@  ........ -%@@@@%=...:+@@@@@%-...   +-.=@@@-. ....... ..@.     .-#@@@@@%+:....-*@@@@@%+:   ...:+%@-                                                   .                  ... ....          .           .       @=......
.  .@-                              .     ..@%@-  +#       .. .=@::%.  .@.  .-@@..=@@@=.....:    .--:..=-%:*-...+@#@-..:%*#+.::....  ..=::.@@.:=%@@@@-..-*@@@@-.......        :-*%@#:.@:.......-.   .... .-=*@@@@@%=. :-+%@@@@@%=.  ...---+#@@=.==          .. .*:-=*%@@@@@@*-..:-=*%@@@@@@#-.                   .                            ....                      .                               @=......
.. .@=                                     .*=@%@.  ..-+  ..  -@..-=  .@  ......=@@:.=@@@+.       .+..+-....@=%:.:@- -@@*@%.....    .+....#@..::+....=%@@@%=..-*@@@@#-       ....  .:@+%%%%*..:  ........==..:@.  .-*%@@@@@%*:.:-*%@@@@@%*-.      :*%#*#*......@. ..        .:=#%@@@@@@%*-. :=*%@@@@@@%%+:.                                   . .                                  .                    @=......
..  @-                                      -#=@%#%.. ...-:+@=-:: ..  @:.--+-*=.*:=+@%:.:@@@%:..-@:.......=*...+*=*--:.:@%@@#-.  . ......*@--+....      -..=@@@@%=:..=@@@@#-:......... ...   .*@%+=:....@-...@.       .....-%@@@@@@%+-.  .-#@@@@@%+-:.. :%..-%@@+=-:.         .....     .:*@@@@@@@#+=:...:=#@@@@@%+=-:.                                        .    .                                   @=. ....
.   .                     .         ..  .   .@-%=@@@:...     .      .@:.:::.......@.  :@@...%@@@=.....   @....#....=-@@-... ..-@@:..... =@:.......    .. .....:%@-%@@@#.  .:%@@@@-....... .......+...=@@@-  *:      --....:@.....    -%@@@@@#:.....:+@@@@@@+..:..    .=@@@--:. . .      . .......%. .-%@@@@@@%=......:=#@@@@@%=-         .    .                ...   .                        .           ......
    @=                             ......   .@:@.@-@*@:..      ..   *:.......   :@     .-+@@-.:*@@@=. .@-....   .=@-=..--*@#=:..+@-%@@:+@:  .....   ... ......@% :    :=%@@@@=...:+@@@@%:...... :@-...... .:=*@@*.........%- =    ..  *-....-=#@@@@@@%-....:-=#@@@@@#--=......:=-=%@%+:.......  +-        ....-=+%@@@@@@@#=...:--=+#@@@@@@+:. .              .........                                . @=......
    @=                               .. ..   .:%*@=.@%@@.         .+:..:%...:- :@.  .....=*:+@@-..+@@@%:...   . ++...-=::..**:@%+--:  .+@@*:..   ............@@.:   .......+%#@@@@%=...:+@@@@@*:.#..:-..-. ....*=.:+%%%%+%#.          =....:=#...:.  :+%@@@@@%#=. ...=#@@@@@@%*:+....-..:###%@+=+   ..*%@-....        ...:+#%@@@@@@%#=.....:=#@@@@@@@*=..-..  ........                       ...        @=......
.  .@-                               ...       :@=@.**@=@-    ...@.--....   . .@..-.=%.==..+:..-@%: .+@@@@-..-==. ......:-@*=:..*@###-:..:=@#@*-............#@       ....-#@:.....:#@@@@#-:...+@@@@@+:.  ...=...@.......:.:*@@*=-.   .:...:=-...:.  ..    ...+-*@@@@@@%+=:.   :*@@@@@@%+-:-:      .+%@#+--:.    .       ........    .:%@@@@@@@%+=-..  .:+#@@@@@@@%=-:..               ... ..  ..        @=......
  ..@=                   .  ..       ...        ..@.=+ @=@@..............@  ..@-..:.::......:- @= -@@...-@@@@-..  ......@%...-%@ .:-.-@@+...%+..:@@+...... %@:    . . .%:@@-........  ...-@@@@#:.....+@@@@%--......:*:.*..     @@:=@@@+=%-:......   ....::  +*  .......:+@@@@@@#:.......:+@@@@@%%....... .. :%@@@=:   ............  .........#+ .-%@@@@@@%+:...:.....-*@@@@@@%: ..  . .......   .   .   @=......
    %-                                .           +=:% -@%-@*.......         @-.........   =*.@+....:+@@:..:#@@@#.....@. ..@.  .-#+*#:  @*=%=*-.....:=#*: =@:   ........=@: . . ..   ..-  ....-+@@@@%:....:=%@@@@#-+-  .-...@+*.--.....:-=*@@#:    ...   . =@...*.   :.. . ...:-=#@@@@@@*-    ..-=*@@@@@@%**.......**-=*%@@+: ..    ... ....-*          ..---=*%@@@@@#:.     .:-=#%@@@@@%*=#  .         %-......
   .#:          .   ..........                    .:.@. @:-%%@-.    +       #+..:@-.-:...    ##-........%@@-. .*@@@@:      ==  ....-=-#@:--*::@#**+.. *+-=@@%-.   .....+@=....    .=..  ....:-....  :#@@@@#-=*@::*@@@@@%+::=+.  :%+::++-+:*.   :+@@@@=... -@. ..-*. .#   +.....    . ..:+%@@@@@@@%=: ....-+%@@@@@@%@-..+.  ..:-+%@%%@#=... .@.        .   .......  .- .:+#@@@@@@%*=:...  .:=#%@%==+-.   #-......
.. .@=                                     .         #+.%+.-@@@+          :........     #-..++  :.:....%% .=@%:...#@@@@-.  .-   .=+....#@%%-:%..-%@:+-.::.....#@@=:...-@@....           ....:=. ... %-....=@@@@@*-. ...-%@@@@%==:@:-=*-.  +++#.=..=#.:.-#@@#+=:.  ........ ...  . . .+. .........-#@@@@@@#=:.  ....-#@@@@@@*-=*....-:...::.-:-=::..  .........    .=     ..........-#@@@@@%#=-:%..@.    @=......
    @=                     . ...                  .   . =#..@+::@.      ....... --  + .:...%@..:. :+ .*%..::..-@@.  .:@@@@=:: .@+....:..  .@@%++@...@#.+%=-* ......#@@%%.....+..    ..  .      .  @%.....=@+..  .+@@@@=     ..*@@@@@-.=.-@*:  :%%@#:++   :=....=@@@-.... ...    .  .: . #........ ........-@@@@@@@*.        .=@@@@@@@+ .:     ....:=#@@@+....             ....  ..      ......--.@.  .  @=......
   .@=                                                  .@  %- +*@@.   ......      ..  . .+%.... *#  =@. .....:%.=@@-...=@@@@%:....  .:=*@....-+@%=#+ -*-..#%#%=..  =@-=%@#-.     ......        --.   ..-@@....+......:+@@@@%-... .-+@@@@@#- ==.=...-#:::.+=.=*-:.. .:=%@@+-..   .+.....:   *.  .. . . ...:@:   .:=*@@@@@@@*=..... ..-=#@@@@@@%-..:*........-=+%@@%:..    . ..              .-@:@-      @=......
.. .@=             .                                     @: =* .-%=@-.....  ..  -   .... -@....... .=@...:..=:. -:..*@@=...:%@@@@: : #:...:%%= ....**#-* ==+:.=%@%#+:-*-   :*%#- ......          ..... .@#:.....:   *- ...:.-#@@@@%=..   .=%@@@@@+::*@=..-#+:.. +: -++=..#@-:+@%@%*:..          ....   ...@-      --.......:+%@@@@@%*=.  .....-#@@@@@@@#-..*  .:*-:..:+%#*%#+.              :@:*=...    @= .....
..  ..              .                           ..          :@..:@-@@#..            ..-:-@-:.@-:.=.-@...        ...  ..+@@-.  .@@@@@=::@@-    .#: .   .#*%+=.==-..-#+%@#:     ..=@@+:.   :.%..   .....:@@    ...:.@=  ....-.     .:%@@@@+-. ....-%@@@@@*-.-@. .:**  ..+%--=#%:..    :%@@#=-.   ..........@*  +* ..-. ... .         .:*@@@@@@#+-:...     -#@@@@@%+=:@. ..=.--=+              %=%-@.....  ........
.   @=         .                            .  . ...        .@:..@=+::@:      .-.   ....%:..    ..-@....:.   .+ ...@......-@@: .@.+@@@@=  :@*   --.%......-=@*+-.#@:. :%%%*@.........*@% ..... .. ....@@.    ...#:....    +:  . ....    .%@@@@#........=@@@@@*.  %-...%@--...#. .%%:....   .*@@@=..  ...+%   .#...:....  .          ..........=%@@@@@+..........  :@=:...+@.:@        .    %*# @:.. .  .@=......
    @=                                                       *%..#@ .+=+@.       .+*     ......  .@:...-  .=:... .- -*..... .=@@@::.:=@@@@%. .#:*=...:*#..%#  :=@-#::@=-...@.+@+%.   +%::+%@*......  @@. .....:.......    .. ...      .@.  ..:=%@@@@%: ...-@:=#@@@@@#:::%. .#*-: ...*: .@@=.......:-#@@@=.  .-+.......        . .....:+.+-..%.      ..-+%@@@@%+.  :%.+#.+@.. @.         ..=#@.:*        @= .....
   .@=            .            ..                            .@...@. .:@.@*   .....   :. -:.... .@...=... :@. .. ..:..#.....  -@*@@+.   -@@@@@-.::.-:...-.-.  +.  =##*==.+%*:.:*@@.=*-:@+. ..-@@@=  =@. .=-......................   :=.......#..  ..=%@@@%**@...:##*@@@@@@*-::--..-#*#    .%..:*#::=.+@:.:#@@@%*:....      . . . ...#:..*.. -      .    .......=#@%:.. .@. .@:@.       ...@=. ..       .@=......
.  .@=                       .                                ..  %-  .@:.-@....     .      ....@-  .*........  -.   :...... .@- ..+@@:  ..-@@@@@=...=.@*   ..*..  ...=@*%++..::@: .-*@.:#=.....  -@@+:+-...........  ..........  -+.. ......* .   ..  .-%%@@@@@%-. .@.  :%@@@@@%=-. =* ..--@-.  .@@:. :-:*:.....=@@@#=:..........-+....-  +-     ...............@%=  :%+  %%.=*       .:@#...@. .    ..@=......
.. .@-                        ..                            ..... ++   %%.@@%%...   ......  ...::.    ........:   .+.  .@-...@-      .=@@:....:%@@%@*+: .:    -..-@....+::--%%@.@...-@*. .@-....  ...::#@@:   ....... . ...   ... ..........:*.   ..   -@..::=@@*%@@@**...::@#:@%@@@@@@--:+==-   .::@=:-##-#..  ..:-  ..:+@@@*:..+=....   . ..  ................#*==.-@+%  @:..%%     .@@:.......       @=......
    -.                   ..                                   .. .-@   -@.:##-@-        ..::.  ......     .....:+.  ... -@:.@*..     ..@:*@@*:....*@@@@%. .=-.*:...-:..@.  .  .-@*%++- ...@@*=.%......... .@@@%:..      ....  .. .............        -@:....++  .-=%%*%@@@@=:..#-  .=*@@@@@@@--@=#-=#..-:.-%=@*..**-  ....:..:=*%@@%-..    ................  .=@ -***.@: -%. ..+%*=##=+*    :#.        -.......
    @=                          .                                 .@:  .@:...+:*@.           .--...=  -.  +  .+...  =@-....@@..    .....=...+@@+    .*@@@@@=.:=-::.....:..%-  :: :. +=-#@**. ..@+#@@+.-   .@+. :*@@*:.... :-.:=... ...........       .@=...=@.     +:......:+@@@@@+. .-@+:-%%@@@@@@%=+#*-+*-.:   +..%-=##%*%-:......  .=*- ......  ........   :@. :@= :%..*-    =-.   -#.              .@=......
    @=            .                                                    .%=    =#.@-....       ...     .. :+. ....    :.   *@   =*. ......@- .+ :@@-:. -@-@@%@@*:....   .%-  .---  ...::%# @ .-#@:::..%%+ :==:#- ....-@@*::-.........    ....... .. ..@*..: :    ...+. .           :%@@@%=:..@.  ..*@@@@@@%-. #:.*+-@-=....+-:#. .-++=:=.+-:.... .. . ... .  ..@: .@-  ++ .@:. .  .   .@.                @=......
    @=                  .                                               +*    :@ .=@....  .                        .  .. *@ ...  .++...   :#    ..-@@=.@@:::@@@@@%.    -     ..:@:....    %@+@+@...  +%@-..%=:%.....    .=@@*.....       .. .     ..#%. .-   .:.   @.              ....:=@@@@@-  ..@@:..-#@@@@@@*.....@:.@= .@=.    =@-..::.......   .      .@+ :@--  @: +%.                            @=......
   .@=                                                                  -%    .@:...*#..  ..:   .                     ..=@    .%+...:. *+   .:.......=@@@:....@%@@#@@%+....-..==..:*:   #......@..#@+++.. .*#-@@#+#   .......-#@@=.     ....       *@..:      -= . .              .  .       :=%@@@@=...-@...:=*@@@@@*=**%=:. ==   +@=..:=%:....           .+#.#+.-* :@ .#.                             @=......
    =:                                                                  .@    .@+..@@#%-       .:   -:  ..             =@:..*:  .-. #%.-+....**.........-@@+. :=  +@@%@@+.....-=....    #:.      :=:.+#  .*%%#:...@:...-=.       .*@@#:     ......-@.          +-            .          .@:   .+....+@@@@%@-    :##:-=@... ....%: =@=...+%:@:              :@+%:..-+ +:                                 +:  ....
.   @=                                                                   .-   .+@. :-=.%@     ..    ....*:      .:  .--@.:+.....=:   ....:@-. -=.          :@@=@- ...@@@@%@*..=*...  .    -.:..=@: . .%@%=++@..  .#*%@:..@           .=@@#-.......@-...         =.          ...  .    .-.     .# .. ..... -%@@@@-. -*@+        ..+@@....%-  @=.           .@+-   .:=.                                  .@=.  ...
..  @=                               .                                         .@.    --.@+.    .. ........ .   :....:@. .:.....-# ...-#   .:@. ..      :    *=@@#:..@=@*@@@#@@:.. .       .  =+. .%:......:@  =@+::.....@.   =@+= ...@@. .:@@%..@-...          .=    .   . . .   .    .   ...:-....  .     ...::#@*..    .. ...%%*#.   @.   @-+ .. ..    %#.@.   -#.                                   @=  ....
    @=                                                           .              @:      +*:@:.........     -%   ... :@............:-:.  .:+.  ..=.  ..... :  .*-:+@@+@+@**.:@@%+@@=   .-*...@%. .=@#:...    :=##+:+*.   .@@%=. .-@=: . .==.....-*@@*.        .:    .. .     ...     **        -- ..   .:-....... .:@.@:      ..@*.@=.  =@     .@#*#+:--:+#%       .:                          .         @=    ..
  ..@=                          . .  .                   .      ..             .#=     .-%..%#....::       ...+. ....................:+.  .=*.  .:*+....    ......  =@@@@*.   :@@@%@@*....-+=. ....*..+:  ...@....=#:+*%*:* ....+@-:.-*+.=:...     .=@@%=. .  . ... ..........    :.          ................   .@+.*%      .@-.:@.. .@-     ...%*%*@-.=%                                              @= .....
 . .@-     ...                                                                  =#      -@...-@-       *........   .........  ..     ..:=.   .%-....:....   .    . :@. .@@%+. .@+.+@@-%@%@-...%-:..@    .:...@.......::  .@.....:@*@:....*#           ...=@@*:.  .   ...  ..   .          .. .....    .. . .     %@...%.    -%  .%%.  :@   .  ..-%.   .:@.                                              @=    ..
 .  .                           .                                               :#..  ...@:....*@       ...:-   .:# ...... .        .... .#-....:..=:..+.         -@.......#@@  @. .:=@@+=@@:....--@      ..:@.....      .@:.%@..-@..... -@   .@@:....  ......*@@-..  .. .   . ..     .... . .  .....         ..=@.. .%=...@#    @= ..=*       ..   ..:@.                                               .. . . .
 ...@=                       .      .                                            ::. ....@%.    .@@:    .       =...:@.       ......  .#....  :=:..-+....:=-     .@-.....  :+.*@@*:::#*.%@@@=@@*   *:-..........-+..   .....-.  ..@..    :@#+-....%-.          ...+%@@-.        :. . .............  .          :@:    ++ .@-    :@   .%=.        . .   .                                                @=......
 .  @=                                                                            .   .  +@    :@@=%:...        .....-= .    *%. *@- .+*....   .- . .=-.    -:. .@-..-.     -+..:@@@%.  :@.:@@#=@@*-....==:.....   :.+    *@......@:.+#%=.+:......*=   .+##+  ...... .:#@@*       .   . ..  .                 .@-.    -%-%..    =#   .@.           .                                                    @= .....
 . .@=                                                                                   .@.   .+-::@#.       . ..      ..        ....  ..      .=..   .%-... ..@- ..     ....%. =..*@@=.=@.  .*@%:*@@-....::::       .+:@=...#:.  .      =+......+#%@=.   *@. .... ..    .@@   .             .               @+.......#.       %=   ..                                                                 @=......
.. .@=                                                                  .                 @:    ..=@:-@:  ....  .       -:        . ..             .%.  ..:#...@+.*.    .......@- . ...=@@+=    .@#@@@-@@-...  .%. ....@@. +:.@.        ..=#..:@@-.=.       @.    .:%+%   .@*.#....           ..    .        #@.....:@@%=      .@.                                                                      @=......
.  .@-                                                                                    %:          .@@....             :*      .                  :-.......@*.  .=:    .    .+-       .+@@+#=...+@+@@*=@@*    ..-..@=..... . -.    .  ..:+:     @- .... .#-.#@#-.  +#....=@*....                  =:     =@.    :@-.=@                                               .                               @-......
..  +:                                                                                    ++         :@.-@#.    .@:          +- ..                   ...+-...%@-:..      ==....---=         .*@@+...#%..-@@%-#@#:...*:....   .@  .**...            -@   .=%@%+..     @%......::-%:                 .       .@:   .%#  ..@.                                                                              +:......
.  .@=                                    .                                               -#         .@:..#@:         +-  .........     ..                ..+@...::..@@%  @:..  ..:*.         ..%@@+.%*     @@*:*@@*-.....   .@ .    -+::.. .       @=@@-...-@......@-    ...@:.@:.*                  .....@-  .%@.:    %-                                    .                                         @=......
..  @=                                                                                    -%.        .@=.  .@*.           .@........   ...         .  ..   -@...+@@#       .==  ....*=:..   .   ...+@@%:    -@%%@@--@@-.   . -#       ...:@:           .:    @. ..-@..      :@   +-..%.                 ..@*. @#. *@    =#                                     .                                        @=......
..  @=                            .                                                      ...-       ..%+     =@-       ..... ...@:.  =+       ......*.  :=-%. ....    .#=    .:*+....%.  =      .. :  =@@@#-...:@*#@@::%@+  .+=.=-.....     :::.......  ..   #* .@@=....    -%    :@@-.+=                %@-@+.          =                                               .                              @=......
   .@=                               .                            .                      .            +#    . .@@....+.... .        :+       .%%..-*@...:-@.    .:+.     +=:  ...-*-..+-  .-: ....     ..*@@+...:@-  =@@=.#@@=......:+.        .:#=:..        .:@=--.....   +*   ....+@%:=+ -: .  :.:..--@:..           ..                                     .                                        @=......
..  ..                                .                   .... .   .                                  :+      ..:@+........              ....     ......:@.        .--.   ..++.... .-+:%.  ..::..    .....::+@@:  @:   .-@@=:=@#....  .+@:.   ......===.      .* .%=....    :.    .     .*@*-:-%:%==+%#:@:               .                                        .                      .              :.......
....@=                                                      .        .                                -@. .   ....:@:...  .          ...-@. ..        ..@-   .       ....    .:%      .@+.... .:-. ........    :@@=@....:+@%@-.@=         ..:... .  ....#@   *@.  @:  .... .@=.              .==.... ..@-                                    .                          .                               @=......
..  @=                                   .                                                            :@.     ...:@@*..   :            ......         .@-              ..*      .=+   ...==.     ..=*..+@-       .=@@%@@+:.:@:.@: +#  ..     .:#:   ......:-@@   .@.    .  .@:               .@:...   %*                                                   . .                  .                      .@=......
... @=                                                                                                .@-   .....@@:.@*                 ......        +-                ...==.    .=-:  .. .==   .  .           .. ..+@@=. :@..@. ...#--..       -==. ....=@@.   =@       .=@.            .  ...   . *@                                                                        .                        @=......
....@=                                                 .                                               @%   . .     .:-@.                           .                        .+.   ...*-.       ..@%*.....    . ..     .:@@-@.-@  ........:.         ..:.+@:..   @#    .   @%                       =@                                                                                                  @=......
    @=                                                                                                 #@..          ...%#.   .*@..            .@-...                          .#:      -#.   .*@.:. ++   ..*.             =@..##.......   .#-.        .@#...  ..@-        @:                       ..                                                         .                                        @-......
.                             .                                                                        :@.              *=@-........ %:        ....=.                           ..=-.     .  .*=:..-*=.    ...:-.         :@-...#*::        ..:*-.    +%.      .:@..      :@.                                                              .                                   .                         .......
   .@=                                                                                                 :#:             .@-.%@:...    .    .-.......-=        .#:               .... .   :@@# -#....   -+:     .. :-   ....@=.....@=   =:       ....-=@=         -@..      -@                 .                                                                                                  .       @=......
.   @-                                                                                                                  @+..:@-.     .    =-....:. .    ........ .          ....::..    ..     .:-.. .  .%+.      .:.....@=. .=...%- ....=-.    ..:#@.          =%....    %@.                                                                                             .                             @=......
..  @=                                                                                                                 .%#....+@     -     .....*    .@....=:.   #=   =. ..#..==@+:.     .#.  .. .-@..    ..#:    ....- @*.. ...  .@:....  .:%   -@:            ==.......  .                  .                                                                                                         @=......
    @=                         ..                          .                                                       .....*%.... .@+       .......     -....=*...   . .%*..::+.  ....  .     .*-      :#-.......-%..   . %@...  ..  ..@-.. ....   #=              +:   .....                             .                      .. .                                    .                         ..      @=  ....
    %-                                                                                        .                    ..  .=@       =@:   ..   ....    ................ ....       . .      .    *=.     .=+...... .*+.  +@    ........:#........%%.              =@=    ....                                       .            ..                                                 .                      @-......
..  -.                        ..    .                   . . ..                                                    ..... .+         @%.....  ..   ....         :+.....           ..              :#........*-.   .   .-@:    ..=*.... .@. ...+@=                .        .                                                  .                                                   .                        =.......
..  @=              ..... .  .  .                .     ....     .                   . ..  .                        .. . .@          :@:..   .     ..          .*....  .. .-#.     .              ..**.......:+. ....-@:       ..:@%.. :%  :@*.                                                                          ....                                                                           .@=......
.  .@=            .  .. .                                                                                               .@-         ..*@           +             .+   ..  .                   .     .-*:... ...:-. .@= .:*=      ..=:. -*+=.                                                                                                                                                           .@=  ....
    @=                                                         .                                                         @=            .@+ ..                     ..           :                  .    :++  ... ...@*. ....:#-    ....++=#                                          .  .    .                                                           .                                              .@=......
    @=            .                                                                                                      +#              *@.        .                 =.                   .             .++     .%# .........::   .=@@:.:.                                                ..                                                                                                           @=......
..  :.                                                                                                                   :@.              .@=      -:    .            .-                  ..                .#   %@   :--...... ..-@+..-= :#      .                                        ..                                                                                                           -.   ...
  . @-                                                                                                                   .@.                 :@-         :                =.             . .                  . =@.     .#......%@.   .@+  -@.        ..                                                                                                                                                @-    ..
..  @=                                                                               .                                   .@:             .  =@.      .       .            :%                ..                 :@: .       .=:@%.           -%       ..                                                                                                                                                 @=... ..
..  @=                                                                                                                    @:.              .:.    ..@:   .  .@..   .          .                               :@:...:..   .+@:        .      --.                                                                                   .                                                                    @=.   ..
..  @=                                                             .                    . .                              -@@:                .=@%:.:.     .@-..               .#-.        ..                 .@=......  -@*.                .-#.                                  ..           .                                                                                                        @= . ...
..  @=                                                    .   . ....                      ...                   . .   ... ....              . ...:#@-.....@=...  .:      . ......   . ..   .            ..  .@* .#... :%-.                                                                                                  .                          .                  .                             @=    ..
                                                          .                                                                  .               ..  ....*@@#:...   :@     .=.... .  .                 .  ..... :+  *@@*.                                                                                                      ..                   .                                                       ..     .
    @=                                                                                                           .                                 @%.  .:%@#=.   .....@:.....@.    .     .    .. ...%=....*@ #:            .                                           .                                                   .                                                                           @-   ...
    @=                                                  .   .                                                                                 .    @@.       .:%@@@%-:..    .@    .@*    ++    @. ..-#@@%.=@.                                                                                                                                         .                                                 @-    ..
    @=                                             .             .                                                                                 #@              ..-*%@@@#*-:....:.....-     .:%%+-....:@:                      .                                                                                                                                   .                                 @=  ....
    @=                                                   .                                                                              .          =@                        ..::-+#%+-:::::....      ..:@-                                                                                              .                                                                                              @=  . ..
    @=                                                                                                                                             .@.                                        *=      ..@+                                                                                                                                                                                              @=     .
                                                                                                                                                   .@.                                       .+.       #=             .                               .                                                                                                                                                      ...
   .*-                                                                                                                                             .@:                   .                 .          %%                                                                                                                                                                                                #:  ....
 .=%@%%-#%%%#+%%%#=#%%%*+%%%%=@%%%**%%%%-%%%%=#%%%#+%%%%=#%%%*=%%%%-@@%%**%%%%:%%%%-%%%%#*%%%%-##%%=*%%%#+%%%%=#%%%*=%%%#=%%%%**%%%%-%%%%-          @:                         .           .         %@.             =%%%%=#%%%*=%%%%=%%%%*#%%%%-%%%%-%%%%**%%%%-%%%%+*%%%#+%%%%+#%%%#-%%%%-%%%%**%%%%=%%%%=*%%%#=%%%%+#%%%#=%%%#-#%%%**%%%%-%%%%=#%%%#=%###+*%%%#-%%%%-%%%%#*%%%%=#%%%*+%%%%=%%%%*=+-=%@@#.  ..
 ...@+..   ..            ...........                ......                        ....             ..   .                     .               .     @-                       .                      =@                                       ... .                .                                                                                                                        ....         @-    ..
  ....                                                                                                            .                                 @=                    .                        .@:                                                                                                                                                                                     .            .     ..
               .                  ..  .                        .                                                                               .    -:  .                            .  .                                                                                                                                                                                                      .              ..
    *#       .:@%..@-.=-#%%%%:%. #+:%@@:-@@@=.          .%..*#%-..%:#:  .*@@@*..   =@:.%+:@%@=.@:   =+   .@@@%.%@%=.                 .         .    %=                      ....                           .                                                        .                                                                                                         .                         @= .....
    #%     ...*@@=:@+.%%. @- .@: @%@@:*@+@.:@=.#@-      .@+@*.-@+@#.@= ..@@...     @@%%@@ :@..:@-   #%   -@-..-@. @=                       .        ##                                                                                                                                                                                                                                                  @=  ....
    #%       .@=*@:@+:%@..@= .@=-@%@%.=@=@@@#  .:.      .@@@-  :@*..@=.  %@--.     @@@@@@ :@: .@-   #%  .-@*=--@@@%                                 +%                                       .                                                                                                                                                                                                          @- .....
..  #%       %@--@+@%-@#:.@+ .@: @#*@:@@=@.:@= %@-     .:@:.@+ .@+..@#=-.@@=-:     @@..@@.+@=:.@*--.%@==--@+=-:@..@#                                +@.                                                                                                                                                                                                                                                 @= .....
....#%          ... .:......         :. .... .  .      ....         .........       . ... ...  ...:...... .... .....                           .    :=.                                                                                                                                                                                                                                                 @=    ..
......              ..                                 .  .               .                   .....                                                     ..                                                                                                      .  .                                                                        .             .                                             ..   ...
    #%        .@@  #@@@-  . ..+@@@*:@@@=.@:   #@.  .-@@.:@@@@:.@@@#:@@@@..+@#.@@@@@+#@@@::@@@#@@:.@=           .@@::@@.=@#@@@@@@* =@-@@@@.+@@@::@@@@ @@@@:.@@@@. @@  *@@@=      -@@@%-@.   .@@. @+ %@+@@@+ @@@@+      %@@=..    .@@@@:-@@@:.@@@@.*@@@+-@    @@@@: -@..-@@@@      .@@@@-@@@@%@=:@@:@@@#@@:.@+=@@@=                                                                                .      @=    ..
    #%        #%@=  -@      ..@- ..@% =@.@:   #@.  .@#@+:@@%@-%# :@-@:-@:.@+@:..@*   +@..@@.=@%%@:@= %@-       -@%@.@@@+@:.@@..@@%@@-@ =@:@=.%@=@::@- +@  %%     ## .@@*.       @-   -@:...#@@@ @+ %@=@ :@-@@@@-     .@@:...    =@    @-.%@%@.-@=@.   -@    @@@@. :@..+@*.      .@%...-@%@%#@@@@@ .@-.@%@:@+ .@:                                                                          . :#          @=  ....
    *%      .:@@@@  -@       .@=...@%.=@.@:   #@. .-@@@@:@-.%@%# :@-@=@@ %@@@%..@#.  +@. @% =@@%=@@= ...   . ..@@%@-@:@%@-.@@..@#.+@-@=@@.@= %@=@@#:  +@  %%         ..=@@      @-   -@. .:@@@@:@+ %@=@.:@-@@... .   @@=@%@.    =@ =@-@- %@@@.-@=@.:@%=@    @=..        .%@.  . .@%:%@:@...#@+#%@ .@- @%=@@*.:@-                                                                            @@*  .      @=   ...
    *%      .@@..@##@@@: .   .+@@@%-@@@+:@@@@-*@@@@@%..@#@@@@+-@@@#:@. @@@: -@..@*..%@@@:.@@@+@% #@+.@@=   .  =@  +%@-.@@-:@@..@* =@-@  @#=@@@--@-.  @@@@:.@@@@.    .@@@@=      -@@@@=@@@@+@  %@+@@@*+@@@* @@@@%     :@@@@@     .@@@@-+@@@-.@@@% +@@@#=@@@@-@@@@=     =@@@@.  ...:@@@@:@@@@%@. %@:@@@#%%.*@*+@@@*                                                                         .@@@@* .      @=   ...
    #%      ...                ...........                 ..........               ..   .           ..              . ......      .                                            ..               .......                             ...               .......             .                ...         ..     .                                                  .                     .@@@@@@@@=      @=..  ..
    ::        ....  .:  .    . ........  ....  .         ............          ... ..  .........  ....-.             .     ... .........           ...........             ......:-. ..... ... ...:......                 .................  .       ...-. .....      .........             ..-.....:.  ...  ..       .          ..  .       .  .  ......  ..                        +@@@@@@@@@@@@@@@:  -.......
.. .%%        @=+@=@@:@*@-:.%#@+::.@@-@%:@+=-.@#*@=     .@*-@*#@:@@::+=.      -@:+@-@#:@+-=@+-=@ .@=@@-@@@@-.@:            :#@.@#.#@=@-=@=@%.@*.:@*::@%--:@%=@@            -@@..@#:+.:-@=:-@-*@-%@-@@:@==@-@% @*         .=@@:@@.:@@:.-+@-.:@-+@:@@...#@-@@::@@:.%@   .-@#:.@+-@-           @*.+.:@*-#.-%@: #%:%@-@:*@.         .@@*.@*@@ =@%@:@@+@@-@@+@-*@.                      .  .#@@@@@@@@@@@=    @= .....
....%%        @++@+@:.%@@#@%@-@@@@-@@%@*:@@@@=@*.%@.    .@@@@: *@@...@@.      -@-*@:.@@=..:@- =@@@@=@+.#@@%@=@:             *@.@# *@=@++@- @@+  .@*..@@@@=@@%@#            @+@% %@@#. .@-.-@%@@:@+ =@:@*#@-.@@*           -@%@#@.@@@@..-@: :@=*@.@@...@#.-@. @@  %@.....@= .@@@@.           %@@*.@@...  %@  #@-@@.-@@:..         @@@+@*@@.=@%*@%@=%@=@@.+@@.                        .     @@@@@@%      .@= .....
 ...#%       .@:  :@*.@%*@%@@ @=   @@ @%:@-...@+.@@     .@- *@..@=. .##       -@     #@.  :@- =@  @=@#.@@@#-@@: -*       .. *@ @# +@-@:.   -@.  .@*..@*.. @%.@@  +=       =@%%@:  .@@.:@- -@ :@:@# #@:@-.   =@   .*:      :@ .=@-@@@@= :@. -@:...@@.  %% *@. @@  %@... .@= .@=.@@.:*.          @@#@:.   #%  %%.....@%   -*       @@.@@#@@.+@%+. @=%%.   .@#                                +@@@:        @=..  ..
   .#%       .@: ..:%@*.:%.** +%%@=*= .%:%@@@:*%%-       #@%*. .#:  .##       :#   ..-#   .%. .*  #:.#@%:%=.-@..@@       -@@%. .#@%..%. ...:%.   #-  #@@@=+= .%:.@@       *-  *-*@@#...%: .*  -= *@%..@:    :%   -@=      .#  :#==  == .%..:@:...*@@@= +@%.  ++. =%%%#:%%@*.%@@*..#@.       *@@#..=@@* *%%#.-=. ...%=   @@       *+ :@=:%@%.=:  +:--     #-                                 *@:         @=......
    +*            .                .                           ..              .   ..                           :.                      .. .          .         .-        .....                       ..     ..  ::                         ...                             ...   :.                                    -               .                                                               @-   ...
                                                                            ..                                                                .                            ...                              ...             ..                     .                                                                                                      .                                                 ....
    %+      ..                    ...              ... ..              ... .                    .                                           . ...                                                                 ..       .                  ...... ..              .              ....                                ....                                                                            @-  ....
 .@@@@@*@@@@@@@@@@@@@@@@@@@@@%@@@@@@@@@@%@@@@@@@@@@@@@@@@@@@@@@@@@@#@@@@@@@@@@*@@@@*@@@@@@@@@@%@@@@@@@@@@@@@@@@@@@@@@@@@@#@@@@@@@@@@*@@@@#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@#@@@@%@@@@@@@@@@%@@@@@@@@@@@@@@@@@@@@@@@@@@%@@@@@@@@@@#@@@@#@@@@@@@@@@%@@@@%@@@@@@@@@@@@@@@@@@@@@%@@@@@@@@@@#@@@@*@@@@@@@@@@%@@@@@@@@@@@@@@@@@@@@@%@@@@*@@@@@@@@@@#@@@@@@@@@@@@@@@@@@@@@#@@@@#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@%@@@@@-.. .
....%+..                      .....                  ..  ......    .. . ..            ........  .   .          .... ...       .      .        .....      .      ...  ...         ...............                  . ..                  ...                                       ......            .  ..                  ..                   ..                                                     .@-   ...
.........        .   .     ... . .  . ........................................ .  ... ..............  .       . . .... ... .......... .  .... ... . ....... .......    ..      .  ................    ... .. ... .........           .  ....   .      ... .          ..       .   ........   ... .    . ....         .    ... .......  ..  ..  .  . .....      .      ..       .             ..     .     ......

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 30)